In [3]:

import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_CSV  = "/kaggle/input/datasets/worldseeker/content/test_results_detailed.csv"
DATASET_JSON = "/kaggle/input/datasets/worldseeker/completecontrastivedataset/siamese_samples_with_srl.json"
OUT_DIR      = "/kaggle/working/results_seed_555"


def load_technique_id_map(json_path: str) -> dict:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    mapping = {}
    for sample in data["samples"]:
        tech_text = (sample.get("Technique_markup", "").strip()
                     or sample.get("Technique_text", ""))
        tid = sample.get("Technique_ID", "")
        if tech_text and tid:
            mapping[tech_text] = tid
    print(f"[Map] Technique_text -> Technique_ID entries: {len(mapping)}")
    return mapping


def is_subtechnique(tid: str) -> bool:
    """T1059.003 -> True,  T1059 -> False"""
    parts = tid.split(".")
    return len(parts) == 2 and parts[1].isdigit()


def per_id_metrics(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for tid, grp in df.groupby("Technique_ID"):
        tp = int(((grp["true_label"] == 1) & (grp["predicted_label"] == 1)).sum())
        fp = int(((grp["true_label"] == 0) & (grp["predicted_label"] == 1)).sum())
        fn = int(((grp["true_label"] == 1) & (grp["predicted_label"] == 0)).sum())
        tn = int(((grp["true_label"] == 0) & (grp["predicted_label"] == 0)).sum())
        n_pos = tp + fn
        prec  = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        rec   = tp / n_pos      if n_pos > 0      else float("nan")
        f1    = (2 * prec * rec / (prec + rec)
                 if not (np.isnan(prec) or np.isnan(rec) or (prec + rec) == 0)
                 else float("nan"))
        records.append({
            "Technique_ID":   tid,
            "is_subtechnique": is_subtechnique(tid),
            "n_positive":     n_pos,
            "n_negative":     tn + fp,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": round(prec, 4),
            "recall":    round(rec,  4),
            "f1":        round(f1,   4),
        })
    return pd.DataFrame(records).sort_values("n_positive", ascending=False)


def recall_bucket_counts(rec_series: pd.Series):
    bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
    labels = ["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
    counts = (pd.cut(rec_series, bins=bins, labels=labels, right=False)
                .value_counts()
                .reindex(labels))
    return counts, labels


def plot_split_histogram(df_tech: pd.DataFrame, df_sub: pd.DataFrame, out_dir: str):
    """Side-by-side histogram: techniques (left) vs sub-techniques (right)."""
    bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
    labels = ["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]

    rec_t = df_tech[df_tech["n_positive"] > 0]["recall"].dropna()
    rec_s = df_sub[df_sub["n_positive"]  > 0]["recall"].dropna()

    ct = pd.cut(rec_t, bins=bins, labels=labels, right=False).value_counts().reindex(labels)
    cs = pd.cut(rec_s, bins=bins, labels=labels, right=False).value_counts().reindex(labels)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

    for ax, counts, title, color, n in [
        (axes[0], ct, "Techniques", "#2980b9", len(rec_t)),
        (axes[1], cs, "Sub-techniques", "#8e44ad", len(rec_s)),
    ]:
        bars = ax.bar(labels, counts, color=color, edgecolor="white", alpha=0.85)
        for bar, val in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.1,
                    str(int(val)), ha="center", va="bottom", fontsize=10)
        ax.set_xlabel("Recall bucket", fontsize=11)
        ax.set_ylabel("Count", fontsize=11)
        ax.set_title(f"{title} (n={n} with ≥1 positive)\nSemanticLink, seed 555",
                     fontsize=11)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    path = os.path.join(out_dir, "per_id_recall_histogram.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[Plot] Saved → {path}")


def print_group_summary(label: str, df: pd.DataFrame):
    df_pos = df[df["n_positive"] > 0]
    rec    = df_pos["recall"].dropna()
    n      = len(df_pos)
    if n == 0:
        print(f"\n  [{label}] No entries with positive instances.")
        return
    counts, labels = recall_bucket_counts(rec)

    print(f"\n{'='*56}")
    print(f"  {label}  (n={n} with ≥1 positive instance)")
    print(f"{'='*56}")
    print(f"  Mean recall   : {rec.mean():.3f}")
    print(f"  Median recall : {rec.median():.3f}")
    print(f"  Std recall    : {rec.std():.3f}")
    n_high = int((rec >= 0.80).sum())
    print(f"  Recall ≥ 0.80 : {n_high}/{n}  ({100*n_high/n:.1f}%)")
    print()
    print("  Recall bucket breakdown:")
    for lbl, cnt in zip(labels, counts):
        pct = 100 * cnt / n if n > 0 else 0
        bar = "█" * int(pct / 5)
        print(f"    {lbl:<10}: {int(cnt):3d}  ({pct:5.1f}%)  {bar}")

    low = df_pos[df_pos["recall"] < 0.40].sort_values("recall")
    print(f"\n  Low-recall (< 0.40): {len(low)}")
    for _, row in low.iterrows():
        print(f"    {row['Technique_ID']:<16}  recall={row['recall']:.2f}"
              f"  n_pos={row['n_positive']}")
    print(f"{'='*56}")




def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    df_test = pd.read_csv(RESULTS_CSV)
    print(f"[Load] {len(df_test)} rows")

    tid_map = load_technique_id_map(DATASET_JSON)
    df_test["Technique_ID"] = df_test["Technique_text"].map(tid_map)

    n_unmapped = df_test["Technique_ID"].isna().sum()
    if n_unmapped:
        print(f"  [Warn] {n_unmapped} unmapped rows dropped")
    df_test = df_test.dropna(subset=["Technique_ID"])

    # Tag sub-techniques
    df_test["is_subtechnique"] = df_test["Technique_ID"].apply(is_subtechnique)
    print(f"\n[Split]  Technique rows    : "
          f"{(~df_test['is_subtechnique']).sum()}")
    print(f"         Sub-technique rows : "
          f"{df_test['is_subtechnique'].sum()}")

    # Compute metrics
    df_metrics = per_id_metrics(df_test)
    df_tech = df_metrics[~df_metrics["is_subtechnique"]]
    df_sub  = df_metrics[ df_metrics["is_subtechnique"]]

    # Save
    df_metrics.to_csv(os.path.join(OUT_DIR, "per_id_metrics_all.csv"),  index=False)
    df_tech.to_csv(   os.path.join(OUT_DIR, "per_technique_metrics.csv"), index=False)
    df_sub.to_csv(    os.path.join(OUT_DIR, "per_subtechnique_metrics.csv"), index=False)
    print(f"\n[Save] CSVs written to {OUT_DIR}")

    # Plot
    plot_split_histogram(df_tech, df_sub, OUT_DIR)

    # Summaries
    print_group_summary("Techniques",     df_tech)
    print_group_summary("Sub-techniques", df_sub)


if __name__ == "__main__":
    main()

[Load] 957 rows
[Map] Technique_text -> Technique_ID entries: 674

[Split]  Technique rows    : 419
         Sub-technique rows : 538

[Save] CSVs written to /kaggle/working/results_seed_555
[Plot] Saved → /kaggle/working/results_seed_555/per_id_recall_histogram.png

  Techniques  (n=36 with ≥1 positive instance)
  Mean recall   : 0.742
  Median recall : 1.000
  Std recall    : 0.425
  Recall ≥ 0.80 : 26/36  (72.2%)

  Recall bucket breakdown:
    0-0.2     :   8  ( 22.2%)  ████
    0.2-0.4   :   1  (  2.8%)  
    0.4-0.6   :   1  (  2.8%)  
    0.6-0.8   :   0  (  0.0%)  
    0.8-1.0   :  26  ( 72.2%)  ██████████████

  Low-recall (< 0.40): 9
    T1491             recall=0.00  n_pos=2
    T1539             recall=0.00  n_pos=2
    T1554             recall=0.00  n_pos=1
    T1212             recall=0.00  n_pos=1
    T1003             recall=0.00  n_pos=1
    T1210             recall=0.00  n_pos=1
    T1546             recall=0.00  n_pos=1
    T1489             recall=0.00  n_pos=1
    